Testing for Lahore


In [2]:
import requests
import sys


sys.path.append(r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset")
from owm_config import API_KEY
city = "Lahore"
url = f"https://api.openweathermap.org/data/2.5/forecast?q={city}&appid={API_KEY}&units=metric"

response = requests.get(url)
data = response.json()

print("Status code:", response.status_code)
print("City confirmed by API:", data.get('city', {}).get('name'))
print("Number of forecast entries returned:", len(data.get('list', [])))

Status code: 200
City confirmed by API: Lahore
Number of forecast entries returned: 40


In [3]:
print(data['list'][0])

{'dt': 1790316000, 'main': {'temp': 32.55, 'feels_like': 33.93, 'temp_min': 32.55, 'temp_max': 32.55, 'pressure': 1007, 'sea_level': 1007, 'grnd_level': 982, 'humidity': 44, 'temp_kf': 0, 'dew_point': 18.72}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'clouds': {'all': 0}, 'wind': {'speed': 0.96, 'deg': 145, 'gust': 0.93}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'd'}, 'dt_txt': '2026-09-25 06:00:00'}


In [4]:
import requests

lat = 31.5204  # Lahore
lon = 74.3587

url = f"https://api.openweathermap.org/data/4.0/onecall/timeline/1h?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"

response = requests.get(url)
data_hourly = response.json()

print("Status code:", response.status_code)
if response.status_code == 200:
    print("Number of hourly entries:", len(data_hourly.get('list', data_hourly.get('data', []))))
    print("\n=== First entry (raw) ===")
    print(data_hourly)
else:
    print("Error:", data_hourly)

Status code: 200
Number of hourly entries: 20

=== First entry (raw) ===
{'lat': 31.5204, 'lon': 74.3587, 'timezone': 'Asia/Karachi', 'timezone_offset': 18000, 'data': [{'dt': 1790308800, 'temp': 30.32, 'feels_like': 33.24, 'pressure': 1008, 'humidity': 59, 'dew_point': 21.41, 'uvi': 0, 'clouds': 0, 'visibility': 10000, 'wind_speed': 2.38, 'wind_deg': 144, 'wind_gust': 2.62, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'pop': 0}, {'dt': 1790312400, 'temp': 30.27, 'feels_like': 32.54, 'pressure': 1008, 'humidity': 56, 'dew_point': 20.52, 'uvi': 0, 'clouds': 0, 'visibility': 10000, 'wind_speed': 1.82, 'wind_deg': 146, 'wind_gust': 1.65, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'pop': 0}, {'dt': 1790316000, 'temp': 30.73, 'feels_like': 32.76, 'pressure': 1008, 'humidity': 53, 'dew_point': 20.05, 'uvi': 0, 'clouds': 0, 'visibility': 10000, 'wind_speed': 0.92, 'wind_deg': 147, 'wind_gust': 0.91, 'weather':

In [5]:
import requests
from datetime import datetime

lat = 31.5204  # Lahore
lon = 74.3587

# Small test range: one week in Jan 2023 (adjust as needed)
start_dt = datetime(2023, 1, 1)
end_dt = datetime(2023, 1, 8)

start_unix = int(start_dt.timestamp())
end_unix = int(end_dt.timestamp())

url = f"http://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start_unix}&end={end_unix}&appid={API_KEY}"

response = requests.get(url)
data_pollution = response.json()

print("Status code:", response.status_code)
if response.status_code == 200:
    print("Number of hourly records:", len(data_pollution.get('list', [])))
    print("\n=== First record (raw) ===")
    print(data_pollution['list'][0])
else:
    print("Error:", data_pollution)

Status code: 200
Number of hourly records: 169

=== First record (raw) ===
{'main': {'aqi': 5}, 'components': {'co': 2429.96, 'no': 8.16, 'no2': 54.84, 'o3': 91.55, 'so2': 34.81, 'pm2_5': 167.11, 'pm10': 198.43, 'nh3': 12.92}, 'dt': 1672556400}


In [6]:
import requests
import pandas as pd
from datetime import datetime
import time

# Your city coordinates (same as used in your Week 8 notebook)
city_coords = {
    'Lahore':     [31.5204, 74.3587],
    'Islamabad':  [33.6844, 73.0479],
    'Faisalabad': [31.4504, 73.1350],
    'Multan':     [30.1575, 71.5249]
}

all_records = []

# Loop through each city
for city, (lat, lon) in city_coords.items():
    print(f"\n=== Downloading pollution data for {city} ===")

    # Loop through each year, but only winter smog months: Nov, Dec, Jan, Feb
    for year in range(2021, 2026):
        for month in [11, 12, 1, 2]:
            # Define the start and end of this month
            start_dt = datetime(year, month, 1)
            if month == 12:
                end_dt = datetime(year + 1, 1, 1)
            else:
                end_dt = datetime(year, month + 1, 1)

            start_unix = int(start_dt.timestamp())
            end_unix = int(end_dt.timestamp())

            url = f"http://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start_unix}&end={end_unix}&appid={API_KEY}"

            response = requests.get(url)

            if response.status_code == 200:
                data = response.json()
                records = data.get('list', [])

                for r in records:
                    all_records.append({
                        'city_name': city,
                        'datetime': datetime.fromtimestamp(r['dt']),
                        'aqi': r['main']['aqi'],
                        'co': r['components']['co'],
                        'no': r['components']['no'],
                        'no2': r['components']['no2'],
                        'o3': r['components']['o3'],
                        'so2': r['components']['so2'],
                        'pm2_5': r['components']['pm2_5'],
                        'pm10': r['components']['pm10'],
                        'nh3': r['components']['nh3'],
                    })

                print(f"  {year}-{month:02d}: {len(records)} records collected")
            else:
                print(f"  {year}-{month:02d}: ERROR {response.status_code} — {response.json()}")

            # Small delay to avoid hitting rate limits
            time.sleep(0.5)

# Convert everything into a DataFrame
pollution_df = pd.DataFrame(all_records)

print(f"\n=== DONE ===")
print(f"Total records collected: {len(pollution_df)}")
print(pollution_df.head())

# Save to CSV
output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\pollution_data_2021_2025.csv"
pollution_df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


=== Downloading pollution data for Lahore ===
  2021-11: 721 records collected
  2021-12: 745 records collected
  2021-01: 721 records collected
  2021-02: 673 records collected
  2022-11: 721 records collected
  2022-12: 673 records collected
  2022-01: 721 records collected
  2022-02: 649 records collected
  2023-11: 721 records collected
  2023-12: 745 records collected
  2023-01: 745 records collected
  2023-02: 625 records collected
  2024-11: 721 records collected
  2024-12: 745 records collected
  2024-01: 721 records collected
  2024-02: 697 records collected
  2025-11: 721 records collected
  2025-12: 738 records collected
  2025-01: 745 records collected
  2025-02: 673 records collected

=== Downloading pollution data for Islamabad ===
  2021-11: 721 records collected
  2021-12: 745 records collected
  2021-01: 721 records collected
  2021-02: 673 records collected
  2022-11: 721 records collected
  2022-12: 673 records collected
  2022-01: 721 records collected
  2022-02: 6

In [7]:
import requests

lat = 31.5204  # Lahore
lon = 74.3587

url = f"http://api.openweathermap.org/data/2.5/air_pollution/forecast?lat={lat}&lon={lon}&appid={API_KEY}"

response = requests.get(url)
data_pollution_forecast = response.json()

print("Status code:", response.status_code)
if response.status_code == 200:
    print("Number of forecast entries:", len(data_pollution_forecast.get('list', [])))
    print("\n=== First entry (raw) ===")
    print(data_pollution_forecast['list'][0])
else:
    print("Error:", data_pollution_forecast)

Status code: 200
Number of forecast entries: 96

=== First entry (raw) ===
{'main': {'aqi': 5}, 'components': {'co': 765.84, 'no': 0.75, 'no2': 5.74, 'o3': 72.38, 'so2': 4.56, 'pm2_5': 102.1, 'pm10': 119.07, 'nh3': 2.14}, 'dt': 1790308800}


In [8]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime
import joblib

model = joblib.load(r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\xgb_model_with_pollution_final.pkl")

city_coords = {
    'Lahore':     [31.5204, 74.3587],
    'Islamabad':  [33.6844, 73.0479],
    'Faisalabad': [31.4504, 73.1350],
    'Multan':     [30.1575, 71.5249]
}

FORECAST_HOURS = 5  

def get_current_visibility(lat, lon):
    """Get current visibility, used as visibility_prev_reading."""
    try:
        url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return r.json()['visibility'] / 1000  # meters -> km
    except Exception as e:
        print(f"  WARNING: current weather API failed ({e}). Using fallback value of 10 km (SAFE default).")
        return 10.0  

def get_weather_forecast_entry(lat, lon):
    """Get the 3-hour forecast entry closest to our target lead time."""
    try:
        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"  ERROR: weather forecast API failed ({e}).")
        return None, None

    now = datetime.now()
    target_time = now + pd.Timedelta(hours=FORECAST_HOURS)

    closest_entry = None
    smallest_diff = None
    for entry in data['list']:
        entry_time = datetime.strptime(entry['dt_txt'], '%Y-%m-%d %H:%M:%S')
        diff = abs((entry_time - target_time).total_seconds())
        if smallest_diff is None or diff < smallest_diff:
            smallest_diff = diff
            closest_entry = entry
            closest_time = entry_time

    return closest_entry, closest_time

def get_pollution_forecast_entry(lat, lon, target_time):
    """Get the hourly pollution forecast entry closest to the SAME target time as the weather entry."""
    try:
        url = f"http://api.openweathermap.org/data/2.5/air_pollution/forecast?lat={lat}&lon={lon}&appid={API_KEY}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"  ERROR: pollution forecast API failed ({e}).")
        return None

    closest_entry = None
    smallest_diff = None
    for entry in data['list']:
        entry_time = datetime.fromtimestamp(entry['dt'])
        diff = abs((entry_time - target_time).total_seconds())
        if smallest_diff is None or diff < smallest_diff:
            smallest_diff = diff
            closest_entry = entry

    return closest_entry

def build_feature_row(city, lat, lon):
    """Build one row of features, in the exact format the model expects.
    Returns (None, None) if forecast data could not be fetched."""
    prev_visibility = get_current_visibility(lat, lon)
    weather_entry, target_time = get_weather_forecast_entry(lat, lon)

    if weather_entry is None:
        return None, None

    pollution_entry = get_pollution_forecast_entry(lat, lon, target_time)

    if pollution_entry is None:
        return None, None

    temp = weather_entry['main']['temp']
    dew = weather_entry['main']['dew_point']
    windspeed_kmh = weather_entry['wind']['speed'] * 3.6   # m/s -> km/h
    windgust_kmh = weather_entry['wind'].get('gust', 0) * 3.6
    winddir = weather_entry['wind'].get('deg', 0)
    pressure = weather_entry['main'].get('sea_level', weather_entry['main']['pressure'])
    cloudcover = weather_entry['clouds']['all']
    precip = weather_entry.get('rain', {}).get('3h', 0)
    snow = weather_entry.get('snow', {}).get('3h', 0)

    row = {
        'temp': temp,
        'humidity': weather_entry['main']['humidity'],
        'dew': dew,
        'windspeed': windspeed_kmh,
        'windgust': windgust_kmh,
        'winddir': winddir,
        'sealevelpressure': pressure,
        'cloudcover': cloudcover,
        'precip': precip,
        'snow': snow,
        'snowdepth': 0,  # not available from API; always ~0 in these cities anyway
        'month': target_time.month,
        'hour': target_time.hour,
        'is_smog_prone_hour': 1 if 4 <= target_time.hour <= 9 else 0,
        'dew_point_depression': temp - dew,
        'city_Faisalabad': 1 if city == 'Faisalabad' else 0,
        'city_Islamabad': 1 if city == 'Islamabad' else 0,
        'city_Lahore': 1 if city == 'Lahore' else 0,
        'city_Multan': 1 if city == 'Multan' else 0,
        'visibility_prev_reading': prev_visibility,
        'winddir_sin': np.sin(np.radians(winddir)),
        'winddir_cos': np.cos(np.radians(winddir)),
        'aqi': pollution_entry['main']['aqi'],
        'co': pollution_entry['components']['co'],
        'no': pollution_entry['components']['no'],
        'no2': pollution_entry['components']['no2'],
        'o3': pollution_entry['components']['o3'],
        'so2': pollution_entry['components']['so2'],
        'pm2_5': pollution_entry['components']['pm2_5'],
        'pm10': pollution_entry['components']['pm10'],
        'nh3': pollution_entry['components']['nh3'],
    }

    return row, target_time

def classify_risk(v):
    if v < 0.5: return "CRITICAL"
    elif v < 2.0: return "HIGH"
    elif v < 5.0: return "MODERATE"
    elif v < 10.0: return "LOW"
    else: return "SAFE"

recommendation_map = {
    "CRITICAL": "Motorway Closure Required",
    "HIGH": "Speed Limit 40 km/h — Caution Required",
    "MODERATE": "Speed Limit 60 km/h — Caution Required",
    "LOW": "Reduced Speed Advised — Monitor Conditions",
    "SAFE": "Normal Operations"
}


results = []

for city, (lat, lon) in city_coords.items():
    print(f"Fetching forecast for {city}...")
    row, target_time = build_feature_row(city, lat, lon)

    if row is None:
        print(f"  SKIPPING {city}: forecast data unavailable. This city will be missing from results.")
        results.append({
            'city': city,
            'forecast_target_time': None,
            'predicted_visibility_km': None,
            'risk_level': 'UNAVAILABLE',
            'recommendation': 'Forecast temporarily unavailable. Please check again shortly.'
        })
        continue

    
    feature_order = ['temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir',
                      'sealevelpressure', 'cloudcover', 'precip', 'snow', 'snowdepth',
                      'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression',
                      'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan',
                      'visibility_prev_reading', 'winddir_sin', 'winddir_cos',
                      'aqi', 'co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']

    X_row = pd.DataFrame([row])[feature_order]
    predicted_visibility = model.predict(X_row)[0]
    risk = classify_risk(predicted_visibility)
    recommendation = recommendation_map[risk]

    results.append({
        'city': city,
        'forecast_target_time': target_time,
        'predicted_visibility_km': round(predicted_visibility, 2),
        'risk_level': risk,
        'recommendation': recommendation
    })

    print(f"  {city}: {predicted_visibility:.2f} km -> {risk}")

results_df = pd.DataFrame(results)
print("\n FORECAST RESULTS (5 hours ahead) ")
print(results_df)

Fetching forecast for Lahore...
  Lahore: 13.12 km -> SAFE
Fetching forecast for Islamabad...
  Islamabad: 20.98 km -> SAFE
Fetching forecast for Faisalabad...
  Faisalabad: 26.71 km -> SAFE
Fetching forecast for Multan...
  Multan: 10.70 km -> SAFE

 FORECAST RESULTS (5 hours ahead) 
         city forecast_target_time  predicted_visibility_km risk_level  \
0      Lahore  2026-09-25 06:00:00                13.120000       SAFE   
1   Islamabad  2026-09-25 06:00:00                20.980000       SAFE   
2  Faisalabad  2026-09-25 06:00:00                26.709999       SAFE   
3      Multan  2026-09-25 06:00:00                10.700000       SAFE   

      recommendation  
0  Normal Operations  
1  Normal Operations  
2  Normal Operations  
3  Normal Operations  


In [9]:
import folium

risk_colors = {
    'CRITICAL': 'red',
    'HIGH': 'orange',
    'MODERATE': 'yellow',
    'LOW': 'lightgreen',
    'SAFE': 'green'
}


lahore_color = risk_colors[results_df[results_df['city']=='Lahore']['risk_level'].values[0]]
islamabad_color = risk_colors[results_df[results_df['city']=='Islamabad']['risk_level'].values[0]]
faisalabad_color = risk_colors[results_df[results_df['city']=='Faisalabad']['risk_level'].values[0]]
multan_color = risk_colors[results_df[results_df['city']=='Multan']['risk_level'].values[0]]

m_forecast = folium.Map(location=[30.8, 72.5], zoom_start=7, tiles='cartodbpositron')

# M-2 — Lahore side
folium.PolyLine(
    locations=[[31.5204, 74.3587], [31.8000, 73.9000], [32.0800, 73.6500], [32.5000, 73.1000]],
    color=lahore_color, weight=6, opacity=0.9,
    tooltip="M-2 (Lahore side)"
).add_to(m_forecast)

# M-2 — Islamabad side
folium.PolyLine(
    locations=[[32.5000, 73.1000], [32.9000, 72.8000], [33.3642, 73.0551]],
    color=islamabad_color, weight=6, opacity=0.9,
    tooltip="M-2 (Islamabad side)"
).add_to(m_forecast)

# M-3 — Lahore to Faisalabad
folium.PolyLine(
    locations=[[31.5204, 74.3587], [31.2000, 73.9000], [30.9500, 73.5500], [30.6000, 73.1000], [30.5450, 72.3114]],
    color=faisalabad_color, weight=6, opacity=0.9,
    tooltip="M-3 (Lahore-Faisalabad)"
).add_to(m_forecast)

# M-4 — Faisalabad to Multan
folium.PolyLine(
    locations=[[30.5450, 72.3114], [30.4000, 72.0000], [30.2000, 71.8000], [29.9500, 71.5500], [30.1575, 71.5249]],
    color=multan_color, weight=6, opacity=0.9,
    tooltip="M-4 (Faisalabad-Multan)"
).add_to(m_forecast)

# 
for city, (lat, lon) in city_coords.items():
    city_data = results_df[results_df['city'] == city].iloc[0]
    risk = city_data['risk_level']
    rec = city_data['recommendation']
    vis = city_data['predicted_visibility_km']
    forecast_time = city_data['forecast_target_time']
    color = risk_colors[risk]

    folium.CircleMarker(
        location=[lat, lon], radius=12, color='black', fill=True,
        fill_color=color, fill_opacity=0.9, tooltip=f"{city}: {risk}",
        popup=folium.Popup(
            f"""<b>{city}</b><br>
            Forecast for: {forecast_time}<br>
            Risk: <b>{risk}</b><br>
            Predicted Visibility: {vis:.2f} km<br>
            Recommendation: {rec}""",
            max_width=280
        )
    ).add_to(m_forecast)

    folium.Marker(
        location=[lat + 0.15, lon],
        icon=folium.DivIcon(html=f'<div style="font-size:12px;font-weight:bold;">{city}</div>')
    ).add_to(m_forecast)

# Legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background-color:white; padding:15px; border:2px solid black;
     border-radius:8px; font-family:Arial;">
<b>Smog Risk Legend (Live 5-Hour Forecast)</b><br>
<span style="color:red;">&#9679;</span> CRITICAL<br>
<span style="color:orange;">&#9679;</span> HIGH<br>
<span style="color:#CCCC00;">&#9679;</span> MODERATE<br>
<span style="color:lightgreen;">&#9679;</span> LOW<br>
<span style="color:green;">&#9679;</span> SAFE
</div>
"""
m_forecast.get_root().html.add_child(folium.Element(legend_html))

# Save
map_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\motorway_risk_map_forecast.html"
m_forecast.save(map_path)
print(f"Live forecast map saved to: {map_path}")

Live forecast map saved to: C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\motorway_risk_map_forecast.html


In [15]:
import os
print(os.getcwd())


C:\Users\Admin\fyp project
